# 실습 2 — Vector Search 2.0 상품 검색 엔진 및 파이프라인 분석

현재 배포 중인 쇼핑 에이전트(`app/embedding_vector.py`)가 런타임에 호출하는 것과 **동일한 API 요청**을, `install.sh`를 통해 사전 프로비저닝된 컬렉션 `amazon-product-768-compact`(768차원 dense 필드 2개 + ScaNN 인덱스 2개)에 직접 실행하고 동작 원리를 확인합니다.

> [!IMPORTANT]
> 이 노트북은 컬렉션과 인덱스를 직접 생성하지 않습니다. 실습을 시작하기 전에 `part2/README.md`의 Cloud Run 배포 명령을 먼저 실행해 주세요.

## 1. 클라이언트 초기화 및 컬렉션 핸들

실제 에이전트 애플리케이션에서 사용하는 4종의 클라이언트를 초기화합니다: 질의 임베딩(`genai`), 벡터/배치 검색(`DataObjectSearch`), 개별 데이터 조회(`DataObject`), 리랭킹(`Rank`).

In [ ]:
import io
import statistics
import urllib.request
from html import escape
from pathlib import Path
from time import perf_counter

import google.auth
from google import genai
from google.genai import types
from google.cloud import vectorsearch_v1beta as vectorsearch
from google.cloud import discoveryengine_v1 as discoveryengine
from IPython.display import HTML, display
from PIL import Image

_, PROJECT_ID = google.auth.default()

# ── app/common.py 와 동일한 값 ──────────────────────────────────────────
LOCATION = "asia-northeast1"
COLLECTION_ID = "amazon-product-768-compact"
COLLECTION_NAME = f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/{COLLECTION_ID}"
IMAGE_SERVER = "https://thumbnail.aidemo.dev"

# ── app/embedding_vector.py 와 동일한 값 ────────────────────────────────
EMBEDDING_MODEL = "gemini-embedding-2"
OUTPUT_DIMENSIONALITY = 768
TEXT_FIELD = "text_embedding"
IMAGE_FIELD = "image_embedding"
SEARCH_TOP_K = 100          # 앱의 SEARCH_TOP_K 와 같은 값입니다.
RANKING_CONFIG = f"projects/{PROJECT_ID}/locations/global/rankingConfigs/default_ranking_config"
TEXT_QUERY_HYBRID_WEIGHTS = [1.35, 0.65]
IMAGE_QUERY_HYBRID_WEIGHTS = [0.65, 1.35]

embedding_client = genai.Client(vertexai=True, project=PROJECT_ID, location="global")
search_client = vectorsearch.DataObjectSearchServiceClient()
data_client = vectorsearch.DataObjectServiceClient()
rank_client = discoveryengine.RankServiceClient()

print("PROJECT_ID :", PROJECT_ID)
print("COLLECTION :", COLLECTION_NAME)

## 2. 백그라운드 인덱싱 완료 확인

`install.sh`에 의해 백그라운드로 실행된 `session2_index_builder.py`는 컬렉션 생성, 데이터 임포트(`ImportDataObjects`), ScaNN 인덱스 2개 생성 등 총 4개의 장기 실행 작업(LRO)을 수행합니다. 현재 작업들의 완료(`done: true`) 여부를 확인합니다.

In [ ]:
!gcloud vector-search operations list --location=asia-northeast1

In [ ]:
# 컬렉션에 실제로 상품이 몇 건 적재되었는지 확인합니다. (벡터 없이 집계만 수행)
try:
    response = search_client.aggregate_data_objects(
        vectorsearch.AggregateDataObjectsRequest(parent=COLLECTION_NAME, aggregate="COUNT")
    )
    # aggregate_results 는 Struct 리스트로 반환되므로 dict 로 변환해야 값을 읽을 수 있습니다.
    rows = [dict(row) for row in response.aggregate_results]
    print("적재된 상품 수 :", rows)
except Exception as exc:  # 임포트가 아직 진행 중이면 여기로 들어옵니다.
    print("❌ 집계 실패:", exc)
    print()
    print("다음 순서로 확인하세요.")
    print("  1) 위 2단계 operations 목록에서 임포트 작업이 done: true 인지 확인")
    print("  2) 터미널에서  tail -30 ~/multimodal-agent/index_builder.log  실행")
    print("  3) 그래도 비어 있으면  bash install.sh  를 다시 실행")


## 3. 컬렉션 스키마 확인 — dense 벡터 필드 2개

`text_embedding`은 상품명과 키워드를 결합한 텍스트 임베딩 필드이며, `image_embedding`은 상품 이미지를 임베딩한 필드입니다.
Gemini Embedding 2는 텍스트와 이미지를 동일한 벡터 공간에 매핑하므로, 하나의 질의 벡터로 두 필드를 모두 효과적으로 검색할 수 있습니다.

In [ ]:
try:
    service_client = vectorsearch.VectorSearchServiceClient()
    collection = service_client.get_collection(name=COLLECTION_NAME)
    print("── data_schema (검색 결과와 함께 받을 수 있는 데이터 필드) ──")
    print(collection.data_schema)
    print("── vector_schema (검색 대상 벡터 필드) ──")
    print(collection.vector_schema)
except Exception as exc:
    print("컬렉션 조회 실패:", exc)

## 4. 상품 카탈로그 프리뷰 및 공용 헬퍼 정의

실습에서 활용할 공용 헬퍼 함수를 정의합니다. (각 함수의 docstring에 실제 애플리케이션 코드 대응 위치가 명시되어 있습니다.)

**카탈로그 데이터셋 안내** — Amazon Berkeley Objects(ABO) 데이터셋 기반 약 10만 건의 상품 카탈로그입니다. 가전·주방·뷰티·식품·가구·신발·가방·주얼리 카테고리 중심이며, **의류(원피스·셔츠 등)는 포함되어 있지 않습니다.** 테스트 질의 시 위 카테고리 범위 내에서 선택해 주시기 바랍니다. (카탈로그에 없는 품목을 질의할 경우 유사도가 낮은 임의의 상품이 매칭될 수 있습니다.)

> 본 컬렉션에는 서버 측 자동 임베딩(`vertex_embedding_config`)이 설정되어 있지 않습니다. 클라이언트가 직접 생성한 벡터를 전달하여 검색하는 `VectorSearch`(bring-your-own-vector) 방식을 사용합니다.

In [ ]:
def embed(text: str | None = None, image: bytes | None = None) -> list[float]:
    """app/embedding_vector.py 의 _embed_with_gemini_embedding_2() 와 동일."""
    contents = text if text is not None else types.Part.from_bytes(data=image, mime_type="image/jpeg")
    response = embedding_client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=contents,
        config=types.EmbedContentConfig(output_dimensionality=OUTPUT_DIMENSIONALITY),
    )
    return list(response.embeddings[0].values)


def to_item(result) -> dict:
    """app/embedding_vector.py 의 _search_result_to_dict() 와 동일."""
    obj = result.data_object
    item_id = obj.data_object_id or obj.name.split("/")[-1]
    return {
        "id": item_id,
        "name": str(obj.data.get("name", "")),
        "description": str(obj.data.get("description", "")),
        "score": result.distance,
    }


def vector_search(embedding, search_field, top_k=SEARCH_TOP_K, metadata_filter=None) -> list[dict]:
    """app/embedding_vector.py 의 _text/_image_similarity_collection_search() 와 동일."""
    clause_kwargs = {
        "search_field": search_field,
        "vector": vectorsearch.DenseVector(values=embedding),
        "top_k": top_k,
        "output_fields": vectorsearch.OutputFields(data_fields=["name", "description"]),
    }
    if metadata_filter is not None:  # 9단계에서 사용합니다.
        clause_kwargs["filter"] = metadata_filter
    request = vectorsearch.SearchDataObjectsRequest(
        parent=COLLECTION_NAME,
        vector_search=vectorsearch.VectorSearch(**clause_kwargs),
    )
    response = search_client.search_data_objects(request)
    return [to_item(result) for result in response.results]


def dedupe(items):
    """같은 상품이 화면을 가득 채우지 않도록 상품명 기준으로 하나만 남깁니다.

    이 카탈로그는 상품 한 개에 사진이 여러 장 있으면 사진마다 별도의 행으로 들어 있고,
    색상·사이즈 변형도 각각 별도의 행입니다. 그래서 검색 상위권은 한두 상품의 사진으로
    가득 찹니다. 검색 자체는 앱과 같은 top_k=100 으로 그대로 두고 화면에 그릴 때만
    걸러냅니다. Part 1에서 만든 비디오 크라우딩 필터와 같은 개념입니다.
    """
    seen, unique = set(), []
    for item in items:
        key = item["name"].strip().lower()
        if key not in seen:
            seen.add(key)
            unique.append(item)
    return unique


def label(name, length):
    """카드에 표시할 상품명을 만듭니다. 앞뒤를 남기고 가운데를 줄입니다.

    이 카탈로그의 상품명은 색상·사이즈가 맨 뒤에 붙습니다. 앞에서부터만 자르면
    변형끼리 완전히 같은 이름으로 보이므로 뒷부분도 함께 남깁니다.
    자르기가 escape() 보다 먼저여야 한다는 점도 중요합니다. 순서가 반대면
    &amp; 나 &#x27; 같은 엔티티가 글자 수를 차지해 이름 뒤쪽이 통째로 사라집니다.
    """
    if len(name) <= length:
        return escape(name)
    head = (length - 1) * 2 // 3
    return escape(name[:head] + "…" + name[head - length + 1:])


def render(items, title="", limit=8):
    """검색 결과를 썸네일 그리드로 표시합니다."""
    items = dedupe(items)
    cards = []
    for rank, item in enumerate(items[:limit], 1):
        cards.append(
            "<div style='width:148px;margin:6px;font-size:11px;text-align:center'>"
            "<img src='{}/{}.webp' style='width:140px;height:140px;object-fit:contain;"
            "background:#fff;border:1px solid #eee'>"
            "<div><b>{}.</b> {}</div><div style='color:#888'>{:.4f}</div></div>".format(
                IMAGE_SERVER, item["id"], rank, label(item["name"], 64), item["score"]
            )
        )
    display(HTML(
        "<b>{}</b><div style='display:flex;flex-wrap:wrap'>{}</div>".format(
            escape(title), "".join(cards))))


PREVIEW_QUERY = "ergonomic office chair with lumbar support"

preview = vector_search(embed(text=PREVIEW_QUERY), TEXT_FIELD)
print("검색 결과 {}건 → 서로 다른 상품 {}종".format(len(preview), len(dedupe(preview))))
render(preview, "카탈로그 프리뷰 — " + PREVIEW_QUERY)

## 5. 텍스트 질의 ➔ `text_embedding` 필드 검색

`find_items` 툴이 전달받은 영어 쿼리를 처리하는 경로입니다. 질의문에 `thermos`·`mug`와 같은 직접적인 단어가 없더라도 보온 텀블러와 진공 보온병이 상위에 검색됩니다.

유사도 점수는 본 컬렉션 기준 **0.65 이상일 경우 적합도가 높으며, 0.58 이하일 경우 카탈로그 내 직접적으로 매칭되는 상품이 적음을 의미**합니다. 전체 카탈로그와의 평균 유사도가 약 0.46 수준이므로, 절대 수치 자체보다 상위 결과와 평균값 간의 상대적 격차에 주목해 주시기 바랍니다.

In [ ]:
QUERIES = [
    "wireless noise cancelling headphones",
    "something that keeps my coffee hot on the desk all morning",
]

text_results = []
for query in QUERIES:
    embed_started = perf_counter()
    query_vector = embed(text=query)
    embed_ms = (perf_counter() - embed_started) * 1000

    search_started = perf_counter()
    results = vector_search(query_vector, TEXT_FIELD)
    search_ms = (perf_counter() - search_started) * 1000

    print("query={!r}  embed_ms={:.1f}  search_ms={:.1f}  결과 {}건 → 상품 {}종".format(
        query, embed_ms, search_ms, len(results), len(dedupe(results))))
    render(results, "text_embedding ← " + query)
    if not text_results:
        text_results = results

## 6. 이미지 질의 ➔ `image_embedding` 필드 검색 (크로스모달)

동일한 이미지 벡터를 ① `image_embedding`(시각적 외형이 유사한 상품)과 ② `text_embedding`(이미지 벡터로 상품 설명문을 매칭하는 크로스모달) 두 필드에 각각 검색합니다. 실제 앱의 카메라 검색 경로가 ①에 해당합니다.

①(이미지 대 이미지)은 0.8대, ②(이미지 대 텍스트)는 0.6대의 유사도 점수를 보입니다. 동일한 상품군이더라도 동일 모달리티(이미지-이미지) 간 비교 시 벡터 거리가 더 가깝게 형성됩니다. 두 경로의 검색 결과 구성이 상호 보완적이므로, 다음 단계에서 RRF(Reciprocal Rank Fusion)를 통한 하이브리드 결합을 적용합니다.

> 질의 이미지는 썸네일 서버의 400px WebP 이미지이며 인덱싱에는 원본 이미지가 사용되었으므로, 질의 상품이 반드시 1위로 나타나지 않을 수 있습니다. 동일 상품의 다른 앵글 사진이나 연관 리스팅이 상위에 노출되는 것이 정상적인 동작입니다.

In [ ]:
def fetch_jpeg(url: str) -> bytes:
    """썸네일을 내려받아 JPEG 바이트로 변환합니다 (앱이 카메라에서 받는 형식과 동일)."""
    with urllib.request.urlopen(url) as response:
        raw = response.read()
    buffer = io.BytesIO()
    Image.open(io.BytesIO(raw)).convert("RGB").save(buffer, format="JPEG")
    return buffer.getvalue()


seed = text_results[0]
seed_url = "{}/{}.webp".format(IMAGE_SERVER, seed["id"])
print("질의 이미지 :", seed["name"])
display(HTML("<img src='{}' width='180' style='border:1px solid #eee'>".format(seed_url)))

image_vector = embed(image=fetch_jpeg(seed_url))
print("이미지 질의 벡터 차원 :", len(image_vector))

render(vector_search(image_vector, IMAGE_FIELD), "① image_embedding ← 이미지 벡터 (생김새)")
render(vector_search(image_vector, TEXT_FIELD), "② text_embedding ← 이미지 벡터 (크로스모달)")

## 7. [핵심] 하이브리드 검색과 RRF 가중치 실험

`batch_search_data_objects`는 여러 검색 절을 단일 요청으로 실행하고 서버 측 RRF 알고리즘을 통해 결과를 융합합니다.

$$\text{score}(d) = \sum_{i} w_i \cdot \frac{1}{k + \text{rank}_i(d)}$$

Part 1에서 직접 계산했던 `alpha`에 해당하는 파라미터가 `weights`이며, **검색 절을 지정한 순서가 곧 가중치 순서**와 일치합니다.

| | 1번 절 (`text_embedding`) | 2번 절 (`image_embedding`) |
| :--- | :--- | :--- |
| `TEXT_QUERY_HYBRID_WEIGHTS` | **1.35** | 0.65 |
| `IMAGE_QUERY_HYBRID_WEIGHTS` | 0.65 | **1.35** |

질의 벡터는 유지한 채 가중치 비율만 조정하여 결과 순위가 어떻게 변동하는지 비교합니다.

In [ ]:
def hybrid_search(embedding, weights, top_k=SEARCH_TOP_K) -> list[dict]:
    """app/embedding_vector.py 의 _hybrid_collection_search() 와 동일한 요청."""
    request = vectorsearch.BatchSearchDataObjectsRequest(
        parent=COLLECTION_NAME,
        searches=[
            vectorsearch.Search(  # 1번 절 → weights[0]
                vector_search=vectorsearch.VectorSearch(
                    search_field=TEXT_FIELD,
                    vector=vectorsearch.DenseVector(values=embedding),
                    top_k=top_k,
                    output_fields=vectorsearch.OutputFields(data_fields=["name", "description"]),
                )
            ),
            vectorsearch.Search(  # 2번 절 → weights[1]
                vector_search=vectorsearch.VectorSearch(
                    search_field=IMAGE_FIELD,
                    vector=vectorsearch.DenseVector(values=embedding),
                    top_k=top_k,
                    output_fields=vectorsearch.OutputFields(data_fields=["name", "description"]),
                )
            ),
        ],
        combine=vectorsearch.BatchSearchDataObjectsRequest.CombineResultsOptions(
            ranker=vectorsearch.Ranker(
                rrf=vectorsearch.ReciprocalRankFusion(weights=weights)
            ),
            output_fields=vectorsearch.OutputFields(data_fields=["name", "description"]),
            top_k=top_k,
        ),
    )
    response = search_client.batch_search_data_objects(request)
    fused = response.results[0].results if response.results else []
    return [to_item(result) for result in fused]


def compare(left_title, left, right_title, right, limit=8):
    """두 결과 목록을 나란히 놓고 순위 변동을 표시합니다."""
    left, right = dedupe(left), dedupe(right)
    left_rank = {item["id"]: i for i, item in enumerate(left, 1)}
    right_rank = {item["id"]: i for i, item in enumerate(right, 1)}

    def badge(item, rank, other):
        previous = other.get(item["id"])
        if previous is None:
            return "<span style='color:#c0392b'>NEW</span>"
        if previous == rank:
            return "<span style='color:#aaa'>=</span>"
        if previous > rank:
            return "<span style='color:#1e8449'>▲{}</span>".format(previous - rank)
        return "<span style='color:#2471a3'>▼{}</span>".format(rank - previous)

    def cells(items, rank, other):
        if rank > len(items):
            return "<td></td><td></td>"
        item = items[rank - 1]
        return ("<td style='padding:4px'><img src='{}/{}.webp' width='52' "
                "style='object-fit:contain;background:#fff'></td>"
                "<td style='padding:4px;font-size:11px'>{} {}</td>").format(
                    IMAGE_SERVER, item["id"], label(item["name"], 46), badge(item, rank, other))

    rows = []
    for rank in range(1, limit + 1):
        rows.append("<tr><td style='padding:4px;color:#888'>{}</td>{}{}</tr>".format(
            rank, cells(left, rank, right_rank), cells(right, rank, left_rank)))
    display(HTML(
        "<table style='border-collapse:collapse'>"
        "<tr><th></th><th colspan='2' style='padding:6px'>{}</th>"
        "<th colspan='2' style='padding:6px'>{}</th></tr>{}</table>".format(
            escape(left_title), escape(right_title), "".join(rows))))


text_weighted = hybrid_search(image_vector, TEXT_QUERY_HYBRID_WEIGHTS)
image_weighted = hybrid_search(image_vector, IMAGE_QUERY_HYBRID_WEIGHTS)

compare(
    "TEXT_QUERY_HYBRID_WEIGHTS = [1.35, 0.65]", text_weighted,
    "IMAGE_QUERY_HYBRID_WEIGHTS = [0.65, 1.35]", image_weighted,
)
print("▲▼ 는 반대편 결과 전체 기준 순위 변동, NEW 는 반대편 결과에 아예 없던 상품입니다.")

> 가중치 중 한쪽을 0으로 설정할 경우(`[2.0, 0.0]` 또는 `[0.0, 2.0]`) 해당 조건이 결과에 반영되지 않아 단일 필드 검색과 동일해집니다. 실제 비즈니스 환경에서는 모달리티 간 상호 보완을 위해 적절한 하이브리드 가중치를 구성합니다.

## 8. Ranking API 리랭킹

벡터 검색은 재현율(recall)을, Ranking API는 정밀도(precision)를 담당합니다. 벡터 검색으로 상위 100건의 후보군을 신속하게 확보한 뒤, Ranking API가 질의문과 상품 설명문을 교차 인코더(cross-encoder)로 정밀하게 재채점하여 최적의 순서로 정렬합니다. `find_items` 툴에서 `ranking_query`를 별도로 전달받는 이유입니다.

In [ ]:
def rank_results(query: str, results: list[dict]) -> list[dict]:
    """app/embedding_vector.py 의 _rank_results() 와 동일 (원본은 리스트를 제자리 정렬)."""
    if not results or not query:
        return results
    records = [
        discoveryengine.RankingRecord(
            id=item["id"], title=item["name"], content=item.get("description", "")
        )
        for item in results
    ]
    response = rank_client.rank(
        request=discoveryengine.RankRequest(
            ranking_config=RANKING_CONFIG,
            query=query,
            records=records,
            top_n=len(records),
        )
    )
    scores = {record.id: record.score for record in response.records}
    ranked = [dict(item, score=scores.get(item["id"], 0.0)) for item in results]
    ranked.sort(key=lambda item: item["score"], reverse=True)
    return ranked


# 6단계의 질의 이미지는 QUERIES[0] 의 검색 결과에서 골랐습니다.
# 리랭킹 질의도 의도가 같아야 순위 변동이 의미를 가집니다.
RANKING_QUERY = QUERIES[0]

reranked = rank_results(RANKING_QUERY, text_weighted)
compare("RRF 융합 직후", text_weighted, "Ranking API 리랭킹 후 — " + RANKING_QUERY, reranked)
print("점수 기준도 함께 바뀝니다: RRF 융합 점수 → Ranking API 관련도 점수(0~1).")

## 9. 에이전트 프롬프트와 `find_items` 툴 호출 흐름

```
경로 A — 카메라 프레임 (자동)
  JPEG → 유사상품 워커 스레드 → _image_similarity_search()
       → image_embedding 단독 검색 [6단계 ①] → 좌측 타일 실시간 갱신

경로 B — 음성 발화 → 툴 호출
  find_items(queries=[영어 쿼리 N개], ranking_query="영어 요약")
    → 쿼리별 스레드 병렬 _collection_search(text=q)   [5단계]
    → id 중복 제거 → _rank_results()                  [8단계]
    → 상위 64건 렌더링 + 음성 브리핑
```

시스템 프롬프트는 카탈로그 텍스트 특성에 맞춰 검색 쿼리를 **영어**로 생성하도록 유도하며, 사용자 대상 음성 응답은 **한국어**로 명확히 출력하도록 지시합니다.

> 애플리케이션은 동일한 `id`를 가진 아이템을 중복 제거합니다. (4단계에서 정의한 `dedupe()`는 노트북 실습용 헬퍼 함수입니다.)

In [ ]:
APP_PROMPT_PY = find_file(
    "app/prompt.py",
    "../part2/app/prompt.py",
)

prompt_source = APP_PROMPT_PY.read_text()
step1 = "## 1단계" + prompt_source.split("## 1단계", 1)[1].split("## 2단계", 1)[0]
print("─" * 78)
print("{}  ::  AGENT_PROMPT 발췌".format(APP_PROMPT_PY))
print("─" * 78)
print(step1.rstrip())

## 실습 2 완료 🎉

1. 질의 코드는 Part 1과 동일한 구조를 유지하며, ScaNN 인덱스를 통해 성능을 가속합니다.
2. 단일 질의 벡터로 `text_embedding`과 `image_embedding` 필드를 모두 유연하게 검색할 수 있습니다.
3. RRF `weights` 설정을 통해 모달리티별 중요도를 조정하고 검색 순위를 최적화할 수 있습니다.
4. Ranking API를 결합하여 1차 검색(재현율) 결과를 정밀도 중심으로 재정렬합니다.
5. 실시간 상호작용 서비스에서는 품질과 지연시간 간의 균형을 고려한 실용적 아키텍처 선택이 중요합니다.

`part2/README.md`로 돌아가 배포 상태를 확인하고, QR 코드로 라이브 쇼핑 에이전트를 직접 체험해 보시기 바랍니다.